In [2]:
import os
import matplotlib.pyplot as plt

import rasterio 
import pandas as pd
import geopandas as gpd
from scipy.ndimage import gaussian_filter
from shapely.geometry import Polygon
from pathlib import Path
from tqdm.auto import tqdm
from rasterio.windows import from_bounds
from rasterio.enums import Resampling

from torch import Tensor

In [3]:
import sys
sys.path.append('../')
from utils import get_tiles_as_gdf, extract_sample, get_window

In [4]:
path = "/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5/labels_grid_5_split_no_masolele.csv"

labels = pd.read_csv(path)


In [16]:
def _get_image_bbox(image_path):


    with rasterio.open(image_path) as src:
        lower_left = src.xy(105, 5, offset='ll')
        upper_left = src.xy(5, 5, offset='ul')

        lower_right = src.xy(105, 105, offset='lr')
        upper_right = src.xy(5, 105, offset='ur')

        #lower_left = src.xy(109, 10, offset='ll')
        #upper_left = src.xy(10, 10, offset='ul')
        #lower_right = src.xy(109, 109, offset='lr')
        #upper_right = src.xy(10, 109, offset='ur')    
        
    poly = Polygon([lower_left, upper_left, upper_right, lower_right])
    
    
    return poly
    
    #print(src.xy(10, 10, offset='ul'))
    
    
    
def resample(src, width, height):

    # resample data to target shape
    data = src.read(
        out_shape=(
            src.count,
            height,
            width
        ),

        resampling=Resampling.nearest
    )

    # scale image transform
    out_transform = src.transform * src.transform.scale(
        (src.width / data.shape[-1]),
        (src.height / data.shape[-2])
    )

    out_profile = src.profile
    out_profile.update(
        {
            'width': width,
            'height': height,
            'transform': out_transform
            }
        )

    return data, out_profile

In [17]:
forestloss_dir = '../../Vancutsem_yearloss'
forestloss_tiles = get_tiles_as_gdf(forestloss_dir)
image_dir = '/home/jan/Documents/EPFL/project_2/data/images/dataset_v5/four_images'
out_dir = '../data/forestloss_tiles/dataset_v5'
target_shape = (33,33)

Path(out_dir).mkdir(parents=True, exist_ok=True)

#shapes = {}
sampleids_to_skip = []

ids = []



In [22]:
skip = True

for idx, row in tqdm(labels.iterrows()):
    #if idx == 29832:
    #    skip = False
        
    #if skip:
    #    continue

    sampleid = str(row.sampleid)
    sampleid_dir = os.path.join(image_dir,sampleid)
    try:
        img_filename = os.listdir(sampleid_dir)[0] 
    except IndexError: #No images downloaded
        sampleids_to_skip.append(int(sampleid))
        continue
    image_path = os.path.join(sampleid_dir, img_filename)
    
    poly = _get_image_bbox(image_path)
    
    bounds = poly.bounds
    
    
    intersecting_tiles = forestloss_tiles.loc[forestloss_tiles.intersects(poly)].filename.values
    
    
    #if len(intersecting_tiles) != 1: #Bordering multiple tiles, skipping
    #    print(sampleid)
    #    sampleids_to_skip.append(int(sampleid))
    
    if len(intersecting_tiles) == 0:
        print(f'does not intersect with any forest loss tile: sampleid {sampleid}')
        continue
    
    tmf_filename = intersecting_tiles[0]

    with rasterio.open(tmf_filename) as src:

        window = from_bounds(*bounds, src.transform)

        # Read the data from the defined window
        data = src.read(window=window)
        
        #if data.shape == (1,33,37):
        #    ids.append(sampleid)
        #if len(intersecting_tiles) != 1:
        #    if data.shape in shapes.keys():
        #        shapes[data.shape] += 1
        #    else:
        #        shapes[data.shape] = 1
        #print(f'intersecting tiles: {len(intersecting_tiles)}', f' patch shape: {data.shape}')
        
            
            
        #min_val = 2000
        #max_val = 2020
        #data[(data < min_val) | (data > max_val)] = 0
        #data[(data >= min_val) & (data <= max_val)] = 1
        #data = data.astype('float')
        #data = gaussian_filter(data, sigma=1)
        #data = Tensor(data)

        # Get the profile (metadata) from the source raster and update it for the output
        profile = src.profile
        profile.update({
            'height': window.height,
            'width': window.width,
            'dtype' : "float32",
            'transform': rasterio.windows.transform(window, src.transform)
        })
        
        
        out_path = os.path.join(out_dir, f'{sampleid}.tif')

        # Write the data to a new raster file
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)
          
        
        #Do it the same way as the Sentinel-2 images processing
        if (data.shape[1], data.shape[2]) != target_shape:
            with rasterio.open(out_path) as src:
                out_image, out_profile = resample(src, target_shape[0], target_shape[1])
            with rasterio.open(out_path, "w", **out_profile) as sink:
                sink.write(out_image)
            
    

0it [00:00, ?it/s]

In [20]:
intersecting_tiles

array(['../../Vancutsem_yearloss/tile_N0_E110.tif'], dtype=object)

In [19]:
len(sampleids_to_skip)

117

In [21]:
print(sampleids_to_skip)

[80117167, 80108061, 100013132, 10135057, 100018532, 10132880, 10134276, 100007277, 100007274, 100007275, 100016957, 100005779, 100015142, 100015140, 100017693, 100017641, 140003200, 140003357, 10108910, 80189828, 140029737, 140033260, 40048875, 40048874, 40047899, 40009769, 40009052, 40031994, 40013122, 40013121, 40002456, 40029757, 40038691, 40040430, 80269387, 80264714, 80271136, 80264713, 80108208, 80211930, 100003239, 100018318, 10132663, 100018562, 10108639, 160008695, 140010401, 140026680, 140009785, 130009808, 130013724, 130009844, 130010221, 130012583, 130009810, 40002743, 40000010, 40003746, 40002457, 40002749, 40036642, 40030727, 150101534, 10630801, 150079713, 150080863, 150100209, 10631177, 110005511, 110003584, 110002736, 110003390, 110003570, 110001473, 110001506, 110001538, 110000144, 110000167, 110000305, 10684181, 10684182, 90077160, 80107147, 80190124, 90025241, 90050982, 90119023, 90099682, 90051518, 90038723, 90038730, 90107266, 90039145, 90158137, 80099880, 801899

In [15]:
labels.loc[labels.sampleid.isin(sampleids_to_skip)].driver.value_counts()

driver
urban                   19
forest regrowth         19
palm oil                15
cocoa                   11
pasture                 11
mining                  10
forest plantation        9
other                    8
shifting agriculture     6
soy                      6
coffee                   2
rubber                   1
Name: count, dtype: int64

In [16]:
labels.loc[labels.sampleid.isin(sampleids_to_skip)].driver.value_counts()

driver
palm oil                29
shifting agriculture    22
forest plantation       22
forest regrowth         13
urban                   12
pasture                 11
soy                      9
cocoa                    7
other                    6
mining                   4
rubber                   4
Name: count, dtype: int64

In [17]:
labels_filtered = labels.loc[~labels.sampleid.isin(sampleids_to_skip)].reset_index(drop=True)
labels_filtered.to_csv(path, index=False)

In [20]:
xx = gpd.GeoDataFrame(labels_filtered)
xx['geometry'] = gpd.GeoSeries.from_wkt(xx['geometry'])
xx = xx.set_geometry("geometry")
xx.to_file('../data/tmp/labels_v4_55k.gpkg')